In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Follow data preprocessing steps given in previous KNN Regr Notebook and get preprocessed data from it

In [3]:
data = pd.read_csv('/content/drive/MyDrive/INM/ML_Feb26/housingDataProcessed.csv')
data

,house_area,house_price,bedrooms,city,floor
0,1201.155608,37.184668,2,Mumbai,ground
1,1184.647009,35.909410,2,Hyd,second
2,1238.824756,40.504743,3,Pune,third
3,986.217155,30.306515,1,Hyd,third
4,1256.945302,38.058359,3,Pune,second
...,...,...,...,...,...
982,1179.215745,37.676472,2,Pune,ground
983,1019.644332,32.719330,1,Pune,second
984,1255.712846,38.081385,3,Pune,first
985,1240.645962,39.179379,3,Hyd,second


In [73]:
from sklearn.model_selection import train_test_split

train_df,test_df = train_test_split(data,test_size=0.25)
train_df.reset_index(drop=True,inplace=True)
test_df.reset_index(drop=True,inplace=True)

print(train_df.shape,test_df.shape)

(740, 5) (247, 5)


## Feature Encoding

In [74]:
train_df.city.value_counts()
# city is nominal variable, and also has not too many unique classes hence we'll do one-hot encoding for this column

,count
city,
Pune,259
Hyd,244
Mumbai,175
Blr,62


In [75]:
from sklearn.preprocessing import OneHotEncoder
city_encoder = OneHotEncoder()

encoded_city = city_encoder.fit_transform(train_df[['city']])
encoded_city = encoded_city.toarray()
encoded_city

array([[0., 1., 0., 0.],
       [0., 0., 0., 1.],
       [0., 1., 0., 0.],
       ...,
       [0., 0., 1., 0.],
       [0., 1., 0., 0.],
       [0., 0., 0., 1.]])

In [76]:
city_encoder.get_feature_names_out()

array(['city_Blr', 'city_Hyd', 'city_Mumbai', 'city_Pune'], dtype=object)

In [77]:
encoded_city_df = pd.DataFrame(encoded_city,columns = city_encoder.get_feature_names_out())
encoded_city_df

,city_Blr,city_Hyd,city_Mumbai,city_Pune
0,0.0,1.0,0.0,0.0
1,0.0,0.0,0.0,1.0
2,0.0,1.0,0.0,0.0
3,0.0,0.0,1.0,0.0
4,0.0,0.0,1.0,0.0
...,...,...,...,...
735,1.0,0.0,0.0,0.0
736,0.0,0.0,0.0,1.0
737,0.0,0.0,1.0,0.0
738,0.0,1.0,0.0,0.0


In [78]:
## concatenate train_df and encoded_city_df horizonatally

train_df = pd.concat([train_df,encoded_city_df],axis=1)
train_df

,house_area,house_price,bedrooms,city,floor,city_Blr,city_Hyd,city_Mumbai,city_Pune
0,1305.790109,41.873703,3,Hyd,third,0.0,1.0,0.0,0.0
1,1167.303584,35.639108,2,Pune,third,0.0,0.0,0.0,1.0
2,1244.259506,38.907785,3,Hyd,third,0.0,1.0,0.0,0.0
3,1094.622288,33.658669,1,Mumbai,third,0.0,0.0,1.0,0.0
4,1303.297925,41.168938,3,Mumbai,first,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...
735,1273.575315,39.887259,3,Blr,third,1.0,0.0,0.0,0.0
736,992.807663,30.694230,1,Pune,third,0.0,0.0,0.0,1.0
737,1143.840962,35.415229,2,Mumbai,ground,0.0,0.0,1.0,0.0
738,1208.744400,37.502332,2,Hyd,first,0.0,1.0,0.0,0.0


In [79]:
train_df.floor.value_counts()

,count
floor,
third,266
second,240
ground,122
first,112


In [80]:
floor_encoder = {'ground':0,'first':1,'second':2,'third':3}
train_df['floor'] = train_df['floor'].apply(lambda x:floor_encoder[x])
train_df

,house_area,house_price,bedrooms,city,floor,city_Blr,city_Hyd,city_Mumbai,city_Pune
0,1305.790109,41.873703,3,Hyd,3,0.0,1.0,0.0,0.0
1,1167.303584,35.639108,2,Pune,3,0.0,0.0,0.0,1.0
2,1244.259506,38.907785,3,Hyd,3,0.0,1.0,0.0,0.0
3,1094.622288,33.658669,1,Mumbai,3,0.0,0.0,1.0,0.0
4,1303.297925,41.168938,3,Mumbai,1,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...
735,1273.575315,39.887259,3,Blr,3,1.0,0.0,0.0,0.0
736,992.807663,30.694230,1,Pune,3,0.0,0.0,0.0,1.0
737,1143.840962,35.415229,2,Mumbai,0,0.0,0.0,1.0,0.0
738,1208.744400,37.502332,2,Hyd,1,0.0,1.0,0.0,0.0


In [81]:
import numpy as np

# log-transformation on house_price
train_df['house_price_log'] = np.log(train_df['house_price'])
train_df

,house_area,house_price,bedrooms,city,floor,city_Blr,city_Hyd,city_Mumbai,city_Pune,house_price_log
0,1305.790109,41.873703,3,Hyd,3,0.0,1.0,0.0,0.0,3.734658
1,1167.303584,35.639108,2,Pune,3,0.0,0.0,0.0,1.0,3.573444
2,1244.259506,38.907785,3,Hyd,3,0.0,1.0,0.0,0.0,3.661194
3,1094.622288,33.658669,1,Mumbai,3,0.0,0.0,1.0,0.0,3.516271
4,1303.297925,41.168938,3,Mumbai,1,0.0,0.0,1.0,0.0,3.717684
...,...,...,...,...,...,...,...,...,...,...
735,1273.575315,39.887259,3,Blr,3,1.0,0.0,0.0,0.0,3.686057
736,992.807663,30.694230,1,Pune,3,0.0,0.0,0.0,1.0,3.424075
737,1143.840962,35.415229,2,Mumbai,0,0.0,0.0,1.0,0.0,3.567142
738,1208.744400,37.502332,2,Hyd,1,0.0,1.0,0.0,0.0,3.624403


In [82]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()

train_df['house_area'] = sc.fit_transform(train_df[['house_area']])
train_df

,house_area,house_price,bedrooms,city,floor,city_Blr,city_Hyd,city_Mumbai,city_Pune,house_price_log
0,1.094246,41.873703,3,Hyd,3,0.0,1.0,0.0,0.0,3.734658
1,-0.330225,35.639108,2,Pune,3,0.0,0.0,0.0,1.0,3.573444
2,0.461342,38.907785,3,Hyd,3,0.0,1.0,0.0,0.0,3.661194
3,-1.077824,33.658669,1,Mumbai,3,0.0,0.0,1.0,0.0,3.516271
4,1.068611,41.168938,3,Mumbai,1,0.0,0.0,1.0,0.0,3.717684
...,...,...,...,...,...,...,...,...,...,...
735,0.762885,39.887259,3,Blr,3,1.0,0.0,0.0,0.0,3.686057
736,-2.125088,30.694230,1,Pune,3,0.0,0.0,0.0,1.0,3.424075
737,-0.571562,35.415229,2,Mumbai,0,0.0,0.0,1.0,0.0,3.567142
738,0.096034,37.502332,2,Hyd,1,0.0,1.0,0.0,0.0,3.624403


In [83]:
features = ['house_area','bedrooms','floor','city_Blr','city_Hyd','city_Mumbai','city_Pune']
x_train = train_df[features]
y_train = train_df['house_price_log']

In [84]:
x_train

,house_area,bedrooms,floor,city_Blr,city_Hyd,city_Mumbai,city_Pune
0,1.094246,3,3,0.0,1.0,0.0,0.0
1,-0.330225,2,3,0.0,0.0,0.0,1.0
2,0.461342,3,3,0.0,1.0,0.0,0.0
3,-1.077824,1,3,0.0,0.0,1.0,0.0
4,1.068611,3,1,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...
735,0.762885,3,3,1.0,0.0,0.0,0.0
736,-2.125088,1,3,0.0,0.0,0.0,1.0
737,-0.571562,2,0,0.0,0.0,1.0,0.0
738,0.096034,2,1,0.0,1.0,0.0,0.0


In [85]:
y_train

,house_price_log
0,3.734658
1,3.573444
2,3.661194
3,3.516271
4,3.717684
...,...
735,3.686057
736,3.424075
737,3.567142
738,3.624403


## Linear Regression Model

In [86]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error as mse,r2_score

In [87]:
lr_model = LinearRegression(fit_intercept=True)

# when fit_intercept=True => hyp function used : y = Wx+b
# when fit_intercept=False => hyp function used : y = Wx

In [88]:
lr_model.fit(x_train.values,y_train)

LinearRegression()

In [89]:
# Evaluate model on train_data

y_pred = lr_model.predict(x_train.values)

In [90]:
def model_evaluation(y_act,y_pred):
  error = mse(y_act,y_pred)
  r2 = r2_score(y_act,y_pred)
  print("MSE =",error)
  print("R2 Score =",r2)

In [91]:
model_evaluation(y_train,y_pred)

MSE = 0.00041027293473257703
R2 Score = 0.9494100742259867


In [93]:
test_df

,house_area,house_price,bedrooms,city,floor
0,1188.449952,36.573499,2,Hyd,third
1,1317.197295,41.665919,3,Pune,third
2,1165.192552,35.645777,2,Pune,second
3,1359.650946,44.369528,3,Blr,third
4,1057.762393,33.442872,1,Pune,third
...,...,...,...,...,...
242,1098.634591,34.089038,1,Pune,second
243,1397.246932,43.387408,3,Hyd,first
244,1192.955621,37.528669,2,Hyd,third
245,1388.595239,44.797857,3,Hyd,second


In [94]:
floor_encoder

{'ground': 0, 'first': 1, 'second': 2, 'third': 3}

In [95]:
encoded_city = city_encoder.transform(test_df[['city']])
encoded_city = encoded_city.toarray()
encoded_city

encoded_city_df = pd.DataFrame(encoded_city,columns = city_encoder.get_feature_names_out())
encoded_city_df

test_df = pd.concat([test_df,encoded_city_df],axis=1)
test_df['floor'] = test_df['floor'].apply(lambda x:floor_encoder[x])

test_df['house_area'] = sc.transform(test_df[['house_area']])

test_df

,house_area,house_price,bedrooms,city,floor,city_Blr,city_Hyd,city_Mumbai,city_Pune
0,-0.112714,36.573499,2,Hyd,3,0.0,1.0,0.0,0.0
1,1.211580,41.665919,3,Pune,3,0.0,0.0,0.0,1.0
2,-0.351939,35.645777,2,Pune,2,0.0,0.0,0.0,1.0
3,1.648258,44.369528,3,Blr,3,1.0,0.0,0.0,0.0
4,-1.456965,33.442872,1,Pune,3,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...
242,-1.036554,34.089038,1,Pune,2,0.0,0.0,0.0,1.0
243,2.034970,43.387408,3,Hyd,1,0.0,1.0,0.0,0.0
244,-0.066369,37.528669,2,Hyd,3,0.0,1.0,0.0,0.0
245,1.945978,44.797857,3,Hyd,2,0.0,1.0,0.0,0.0


In [96]:
test_df['house_price_log']=np.log(test_df['house_price'])
test_df

,house_area,house_price,bedrooms,city,floor,city_Blr,city_Hyd,city_Mumbai,city_Pune,house_price_log
0,-0.112714,36.573499,2,Hyd,3,0.0,1.0,0.0,0.0,3.599324
1,1.211580,41.665919,3,Pune,3,0.0,0.0,0.0,1.0,3.729684
2,-0.351939,35.645777,2,Pune,2,0.0,0.0,0.0,1.0,3.573631
3,1.648258,44.369528,3,Blr,3,1.0,0.0,0.0,0.0,3.792553
4,-1.456965,33.442872,1,Pune,3,0.0,0.0,0.0,1.0,3.509839
...,...,...,...,...,...,...,...,...,...,...
242,-1.036554,34.089038,1,Pune,2,0.0,0.0,0.0,1.0,3.528976
243,2.034970,43.387408,3,Hyd,1,0.0,1.0,0.0,0.0,3.770169
244,-0.066369,37.528669,2,Hyd,3,0.0,1.0,0.0,0.0,3.625105
245,1.945978,44.797857,3,Hyd,2,0.0,1.0,0.0,0.0,3.802160


In [97]:
x_test = test_df[features]
y_test = test_df['house_price_log']
y_pred = lr_model.predict(x_test.values)
model_evaluation(y_test,y_pred)

MSE = 0.00046467546095939826
R2 Score = 0.9516710827647104
